In [0]:
USE CATALOG projectcatalog

In [0]:
CREATE SCHEMA Gold_Schema

In [0]:
USE SCHEMA Gold_Schema

In [0]:
SELECT * FROM sliverschemasales.tblprjcustomersilver LIMIT 10;

customer_id,first_name,last_name,fullname,email,city,state,created_timesatmp,sourcefile
C01965,Chad,Martinez,Chad Martinez,aaron30@gmail.com,Rojasview,FL,2026-08-11T10:23:27.588Z,customersfile.parquet
C00410,Justin,Henry,Justin Henry,aaron67@yahoo.com,Andrewburgh,WV,2026-08-11T10:23:27.588Z,customersfile.parquet
C01903,Beth,Smith,Beth Smith,aaron98@gmail.com,Gallegosport,AK,2026-08-11T10:23:27.588Z,customersfile.parquet
C01683,Stephanie,Peterson,Stephanie Peterson,aarongriffith@moore.com,Ericfort,SC,2026-08-11T10:23:27.588Z,customersfile.parquet
C01490,Christopher,Guerra,Christopher Guerra,aaronjackson@hill-smith.com,East Kaylamouth,VA,2026-08-11T10:23:27.588Z,customersfile.parquet
C00119,Katie,Davis,Katie Davis,aaronroberts@hotmail.com,Lake Robertland,KS,2026-08-11T10:23:27.588Z,customersfile.parquet
C01804,Hannah,Rogers,Hannah Rogers,abarajas@hotmail.com,New Davidview,KY,2026-08-11T10:23:27.588Z,customersfile.parquet
C00451,Evelyn,Wolfe,Evelyn Wolfe,abigail73@jones.info,Craigfurt,WV,2026-08-11T10:23:27.588Z,customersfile.parquet
C01065,Alexander,Lara,Alexander Lara,abigailbanks@hotmail.com,South Amy,IL,2026-08-11T10:23:27.588Z,customersfile.parquet
C00198,Autumn,Williams,Autumn Williams,abigailray@perez.com,New Josephhaven,WA,2026-08-11T10:23:27.588Z,customersfile.parquet


In [0]:
CREATE TABLE IF NOT EXISTS dim_customer
(
    customer_key   BIGINT NOT NULL, --SURROGATE KEY
    customer_id    STRING NOT NULL,
    first_name     STRING,
    last_name      STRING,
    fullname      STRING,
    email          STRING,
    email_domain   STRING,
    city           STRING,
    state          STRING,
    customer_location STRING,
    source_created_datetime TIMESTAMP,
    sourcefile      STRING,
    gold_created_datetime TIMESTAMP NOT NULL,
    gold_updated_datetime TIMESTAMP NOT NULL
) USING DELTA

In [0]:
CREATE OR REPLACE TEMP VIEW vw_customer_gold_source AS
SELECT
    XXHASH64(customer_id)             AS customer_key,
    customer_id,
    first_name,
    last_name,
    fullname                          ,
    email,
    LOWER(SPLIT(email, '@')[1])       AS email_domain,
    city,
    state,
    CONCAT_WS(', ', city, state)      AS customer_location,
    created_timesatmp                 AS source_created_datetime,
    sourcefile                       ,
    current_timestamp()               AS load_datetime
FROM
(
    SELECT * , ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY created_timesatmp DESC, email) 
    AS
    row_num FROM sliverschemasales.tblprjcustomersilver
) src WHERE row_num = 1

In [0]:
select * from vw_customer_gold_source

customer_key,customer_id,first_name,last_name,fullname,email,email_domain,city,state,customer_location,source_created_datetime,sourcefile,load_datetime
2735230679661891639,C00001,Emily,Mooney,Emily Mooney,rushjeff@ryan.org,ryan.org,Johnsonmouth,MS,"Johnsonmouth, MS",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-8558369134346332286,C00002,Andrea,Sellers,Andrea Sellers,mccoykiara@kelly.com,kelly.com,Stephenfort,WY,"Stephenfort, WY",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-6558032996247525554,C00003,Craig,Hayes,Craig Hayes,rebeccamiller@yahoo.com,yahoo.com,South Stephenshire,LA,"South Stephenshire, LA",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
7321628704337629207,C00004,Bryan,Scott,Bryan Scott,lawrence05@campbell.info,campbell.info,Chrisland,ND,"Chrisland, ND",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-7536618453668114900,C00005,Sean,Vasquez,Sean Vasquez,carrie45@yahoo.com,yahoo.com,East Dennistown,RI,"East Dennistown, RI",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-3979898635484049401,C00006,Kevin,Mccarthy,Kevin Mccarthy,traceyramos@gmail.com,gmail.com,North Matthew,IN,"North Matthew, IN",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-5317803948870611650,C00007,Amanda,Doyle,Amanda Doyle,scottallen@gmail.com,gmail.com,Joneshaven,VA,"Joneshaven, VA",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-5614710278438779986,C00008,Paul,Campos,Paul Campos,sullivanjeremy@horton-adams.com,horton-adams.com,South Nathanfurt,CT,"South Nathanfurt, CT",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
-4044674756901600687,C00009,Mary,Green,Mary Green,dennis03@yahoo.com,yahoo.com,Kimberlyview,MD,"Kimberlyview, MD",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z
6631846047666284764,C00010,James,Myers,James Myers,charles58@murillo.net,murillo.net,West Hector,OK,"West Hector, OK",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:38:47.443Z


In [0]:
DESC dim_customer

col_name,data_type,comment
customer_key,bigint,null
customer_id,string,null
first_name,string,null
last_name,string,null
fullname,string,null
email,string,null
email_domain,string,null
city,string,null
state,string,null
customer_location,string,null


In [0]:
MERGE INTO dim_customer AS target
USING vw_customer_gold_source AS source
ON target.customer_id  = source.customer_id
WHEN MATCHED AND
(
    NOT(target.first_name         <=> source.first_name )
    OR NOT(target.last_name         <=> source.last_name )
    OR NOT (target.fullname        <=> source.fullname)
    OR NOT (target.email            <=> source.email)
    OR NOT (target.email_domain     <=> source.email_domain)
    OR NOT (target.city            <=> source.city)
    OR NOT (target.state           <=> source.state)
    OR NOT (target.customer_location <=> source.customer_location)
    OR NOT (target.sourcefile      <=> source.sourcefile)
)
THEN UPDATE SET
    target.first_name                = source.first_name,
    target.last_name                 = source.last_name,
    target.fullname                 = source.fullname,
    target.email                     = source.email,
    target.email_domain              = source.email_domain,
    target.city                      = source.city,
    target.state                     = source.state,
    target.customer_location         = source.customer_location,
    target.source_created_datetime   = source.source_created_datetime,
    target.sourcefile               = source.sourcefile,
    target.gold_updated_datetime     = source.load_datetime

WHEN NOT MATCHED
THEN INSERT
(
    customer_key,
    customer_id,
    first_name,
    last_name,
    fullname,
    email,
    email_domain,
    city,
    state,
    customer_location,
    source_created_datetime,
    sourcefile,
    gold_created_datetime,
    gold_updated_datetime
)
VALUES
(
    source.customer_key,
    source.customer_id,
    source.first_name,
    source.last_name,
    source.fullname,
    source.email,
    source.email_domain,
    source.city,
    source.state,
    source.customer_location,
    source.source_created_datetime,
    source.sourcefile,
    source.load_datetime,
    source.load_datetime
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
select * from dim_customer

customer_key,customer_id,first_name,last_name,fullname,email,email_domain,city,state,customer_location,source_created_datetime,sourcefile,gold_created_datetime,gold_updated_datetime
2735230679661891639,C00001,Emily,Mooney,Emily Mooney,rushjeff@ryan.org,ryan.org,Johnsonmouth,MS,"Johnsonmouth, MS",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-8558369134346332286,C00002,Andrea,Sellers,Andrea Sellers,mccoykiara@kelly.com,kelly.com,Stephenfort,WY,"Stephenfort, WY",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-6558032996247525554,C00003,Craig,Hayes,Craig Hayes,rebeccamiller@yahoo.com,yahoo.com,South Stephenshire,LA,"South Stephenshire, LA",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
7321628704337629207,C00004,Bryan,Scott,Bryan Scott,lawrence05@campbell.info,campbell.info,Chrisland,ND,"Chrisland, ND",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-7536618453668114900,C00005,Sean,Vasquez,Sean Vasquez,carrie45@yahoo.com,yahoo.com,East Dennistown,RI,"East Dennistown, RI",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-3979898635484049401,C00006,Kevin,Mccarthy,Kevin Mccarthy,traceyramos@gmail.com,gmail.com,North Matthew,IN,"North Matthew, IN",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-5317803948870611650,C00007,Amanda,Doyle,Amanda Doyle,scottallen@gmail.com,gmail.com,Joneshaven,VA,"Joneshaven, VA",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-5614710278438779986,C00008,Paul,Campos,Paul Campos,sullivanjeremy@horton-adams.com,horton-adams.com,South Nathanfurt,CT,"South Nathanfurt, CT",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-4044674756901600687,C00009,Mary,Green,Mary Green,dennis03@yahoo.com,yahoo.com,Kimberlyview,MD,"Kimberlyview, MD",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
6631846047666284764,C00010,James,Myers,James Myers,charles58@murillo.net,murillo.net,West Hector,OK,"West Hector, OK",2026-08-11T10:23:27.588Z,customersfile.parquet,2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z


In [0]:
CREATE OR REPLACE VIEW vw_customer_business AS
SELECT 
    customer_key,
    customer_id,
    first_name,
    last_name,
    fullname,
    email,
    email_domain,
    city,
    state,
    customer_location,
    gold_created_datetime,
    gold_updated_datetime
    FROM
    dim_customer

In [0]:
SELECT * FROM vw_customer_business

customer_key,customer_id,first_name,last_name,fullname,email,email_domain,city,state,customer_location,gold_created_datetime,gold_updated_datetime
2735230679661891639,C00001,Emily,Mooney,Emily Mooney,rushjeff@ryan.org,ryan.org,Johnsonmouth,MS,"Johnsonmouth, MS",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-8558369134346332286,C00002,Andrea,Sellers,Andrea Sellers,mccoykiara@kelly.com,kelly.com,Stephenfort,WY,"Stephenfort, WY",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-6558032996247525554,C00003,Craig,Hayes,Craig Hayes,rebeccamiller@yahoo.com,yahoo.com,South Stephenshire,LA,"South Stephenshire, LA",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
7321628704337629207,C00004,Bryan,Scott,Bryan Scott,lawrence05@campbell.info,campbell.info,Chrisland,ND,"Chrisland, ND",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-7536618453668114900,C00005,Sean,Vasquez,Sean Vasquez,carrie45@yahoo.com,yahoo.com,East Dennistown,RI,"East Dennistown, RI",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-3979898635484049401,C00006,Kevin,Mccarthy,Kevin Mccarthy,traceyramos@gmail.com,gmail.com,North Matthew,IN,"North Matthew, IN",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-5317803948870611650,C00007,Amanda,Doyle,Amanda Doyle,scottallen@gmail.com,gmail.com,Joneshaven,VA,"Joneshaven, VA",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-5614710278438779986,C00008,Paul,Campos,Paul Campos,sullivanjeremy@horton-adams.com,horton-adams.com,South Nathanfurt,CT,"South Nathanfurt, CT",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
-4044674756901600687,C00009,Mary,Green,Mary Green,dennis03@yahoo.com,yahoo.com,Kimberlyview,MD,"Kimberlyview, MD",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z
6631846047666284764,C00010,James,Myers,James Myers,charles58@murillo.net,murillo.net,West Hector,OK,"West Hector, OK",2026-09-05T06:50:18.527Z,2026-09-05T06:50:18.527Z


In [0]:
CREATE OR REPLACE TABLE agg_customer_by_state
USING DELTA
COMMENT 'Customer Summary By State'
AS
SELECT 
    state,
    COUNT(*)  AS TOTAL_CUSTOMERS,
    COUNT(DISTINCT city) AS TOTAL_CITIES,
    COUNT(DISTINCT email) AS UNIQUE_EMAILS,
    current_timestamp AS refreshed_datetime
FROM
dim_customer
GROUP BY state

num_affected_rows,num_inserted_rows


In [0]:
select * from agg_customer_by_state

state,TOTAL_CUSTOMERS,TOTAL_CITIES,UNIQUE_EMAILS,refreshed_datetime
NJ,33,33,33,2026-09-05T07:47:38.543Z
IL,38,38,38,2026-09-05T07:47:38.543Z
IN,38,38,38,2026-09-05T07:47:38.543Z
WY,43,43,43,2026-09-05T07:47:38.543Z
OK,44,44,44,2026-09-05T07:47:38.543Z
CA,41,41,41,2026-09-05T07:47:38.543Z
WV,37,37,37,2026-09-05T07:47:38.543Z
MN,43,43,43,2026-09-05T07:47:38.543Z
CT,40,40,40,2026-09-05T07:47:38.543Z
ND,31,31,31,2026-09-05T07:47:38.543Z


In [0]:
CREATE OR REPLACE TABLE agg_customer_by_city
USING DELTA
COMMENT 'Customer By City and State'
AS
SELECT
   state,
   city,
   CONCAT_WS(', ',city, state) AS customer_location,
   COUNT(*) AS total_customers,
   current_timestamp() AS refreshed_datetime
FROM
dim_customer
GROUP BY state, city


num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM agg_customer_by_city

state,city,customer_location,total_customers,refreshed_datetime
WY,Stephenfort,"Stephenfort, WY",1,2026-09-05T07:53:54.434Z
ND,Chrisland,"Chrisland, ND",1,2026-09-05T07:53:54.434Z
WI,East Jenniferview,"East Jenniferview, WI",1,2026-09-05T07:53:54.434Z
OR,Kaiserhaven,"Kaiserhaven, OR",1,2026-09-05T07:53:54.434Z
AK,Allisonhaven,"Allisonhaven, AK",1,2026-09-05T07:53:54.434Z
CO,Lopezfurt,"Lopezfurt, CO",2,2026-09-05T07:53:54.434Z
HI,Shanehaven,"Shanehaven, HI",1,2026-09-05T07:53:54.434Z
TN,Kerrhaven,"Kerrhaven, TN",1,2026-09-05T07:53:54.434Z
MA,Randallchester,"Randallchester, MA",1,2026-09-05T07:53:54.434Z
KY,Seanmouth,"Seanmouth, KY",1,2026-09-05T07:53:54.434Z


In [0]:
CREATE OR REPLACE TABLE agg_customer_by_email_domain
USING DELTA
COMMENT 'Customer Summary By email domain'
AS
SELECT 
    email_domain,
    COUNT(*) AS total_customers,
    ROUND
    (
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2
    ) AS customer_percentage,
    current_timestamp() AS refreshed_datetime
FROM
dim_customer
GROUP BY email_domain

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM agg_customer_by_email_domain

email_domain,total_customers,customer_percentage,refreshed_datetime
king.com,3,0.15,2026-09-05T08:02:03.154Z
villanueva.org,1,0.05,2026-09-05T08:02:03.154Z
olsen.com,1,0.05,2026-09-05T08:02:03.154Z
lopez-lawrence.net,1,0.05,2026-09-05T08:02:03.154Z
willis-anderson.com,1,0.05,2026-09-05T08:02:03.154Z
scott-wall.com,1,0.05,2026-09-05T08:02:03.154Z
taylor.com,2,0.10,2026-09-05T08:02:03.154Z
harrison.com,1,0.05,2026-09-05T08:02:03.154Z
ramsey-palmer.info,1,0.05,2026-09-05T08:02:03.154Z
payne.info,1,0.05,2026-09-05T08:02:03.154Z


In [0]:
CREATE OR REPLACE VIEW vw_customer_state_ranking  AS
SELECT
    state,
    total_customers,
    total_cities,
    unique_emails,
    DENSE_RANK() OVER
    (
        ORDER BY total_customers DESC
    ) AS state_rank
FROM
agg_customer_by_state

In [0]:
SELECT * FROM vw_customer_state_ranking

state,total_customers,total_cities,unique_emails,state_rank
IA,49,49,49,1
VT,49,49,49,1
ME,47,47,47,2
GA,46,45,46,3
NV,45,45,45,4
KY,45,45,45,4
OK,44,44,44,5
TN,44,44,44,5
MT,44,44,44,5
FL,43,43,43,6


In [0]:
CREATE OR REPLACE VIEW vw_customer_city_ranking AS
SELECT
    state,
    city,
    customer_location,
    total_customers,
    DENSE_RANK() OVER
    (
        PARTITION BY state 
        ORDER BY total_customers DESC
    ) AS CITY_RANK
FROM agg_customer_by_city

In [0]:

SELECT * FROM vw_customer_city_ranking

state,city,customer_location,total_customers,CITY_RANK
AK,Allisonhaven,"Allisonhaven, AK",1,1
AK,Chaneyton,"Chaneyton, AK",1,1
AK,Harrisonmouth,"Harrisonmouth, AK",1,1
AK,Lake Craig,"Lake Craig, AK",1,1
AK,East Caitlyn,"East Caitlyn, AK",1,1
AK,Lake Ruthview,"Lake Ruthview, AK",1,1
AK,Gonzalesstad,"Gonzalesstad, AK",1,1
AK,Dickersonport,"Dickersonport, AK",1,1
AK,New William,"New William, AK",1,1
AK,Port Jenniferborough,"Port Jenniferborough, AK",1,1


In [0]:
OPTIMIZE dim_customer ZORDER BY (customer_id, state, city);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 97407), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1788606103649, 1788606104984, 8, 0, null, List(0, 0), null, 14, 14, 0, 0, null, null, 0)"
